# Final-evaluation rerun: Decision Tree

This notebook reruns the established tuning procedure on the **new frozen development split** created by notebook 10. It never loads the locked final-test rows. It selects validation thresholds for 70%, 75%, 80%, 85%, and 90% target recall, then saves the frozen model and artifacts for notebook 17.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

In [2]:
import json
import joblib

In [3]:
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = (
        parent
        / "Processed_Dataset"
        / "diabetic_data_cleaned_stage1.csv"
    )

    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find "
        "Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )


OUTPUT_DIR = (
    PROJECT_ROOT
    / "Model_Results"
    / "decision_tree_optimisation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


df = pd.read_csv(DATA_PATH)

print("Dataset path:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

df.head()

# Final-evaluation artifacts are kept separate from the earlier development runs.
FINAL_EVALUATION_DIR = PROJECT_ROOT / "Final_Evaluation"
SPLIT_DIR = FINAL_EVALUATION_DIR / "Data_Splits"
OUTPUT_DIR = FINAL_EVALUATION_DIR / "Model_Artifacts" / "decision_tree"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nFinal-evaluation output directory:")
print(OUTPUT_DIR)


Dataset path:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Processed_Dataset/diabetic_data_cleaned_stage1.csv

Dataset shape:
(69987, 56)

Final-evaluation output directory:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Final_Evaluation/Model_Artifacts/decision_tree


In [4]:
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = (
    categorical_features
    + numeric_features
)


missing_features = [
    feature
    for feature in model_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        f"These modelling features are missing: "
        f"{missing_features}"
    )


X = df[model_features].copy()
y = df[target_col].astype(int).copy()


print("X shape:")
print(X.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget counts:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))

X shape:
(69987, 18)

Features used:
['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target counts:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


In [5]:
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = (
    forbidden_features
    .intersection(X.columns)
)

assert not unexpected_features, (
    "Unexpected or potentially leaking features found: "
    f"{unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names were found in X."
)

assert len(X) == len(y), (
    "X and y contain different numbers of rows."
)

assert y.isna().sum() == 0, (
    "The target contains missing values."
)

assert set(y.unique()).issubset({0, 1}), (
    "The target must contain only 0 and 1."
)

print("Feature and target checks passed.")

Feature and target checks passed.


In [6]:

# IMPORTANT: the final-test rows are deliberately NOT loaded in this notebook.
# Notebook 10 creates one fixed split that every model reuses.

SPLIT_DIR = PROJECT_ROOT / "Final_Evaluation" / "Data_Splits"

train_split_path = SPLIT_DIR / "model_train_rows.csv"
validation_split_path = SPLIT_DIR / "threshold_validation_rows.csv"

if not train_split_path.exists() or not validation_split_path.exists():
    raise FileNotFoundError(
        "Final split files are missing. Run 10_create_final_split.ipynb first."
    )

train_idx = (
    pd.read_csv(train_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)
val_idx = (
    pd.read_csv(validation_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)

assert set(train_idx).isdisjoint(set(val_idx))

X_model_train = X.iloc[train_idx].copy()
y_model_train = y.iloc[train_idx].copy()

X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

split_summary = pd.DataFrame({
    "split": ["model_training", "threshold_validation"],
    "rows": [len(X_model_train), len(X_val)],
    "positive_count": [int(y_model_train.sum()), int(y_val.sum())],
    "positive_rate": [float(y_model_train.mean()), float(y_val.mean())],
})

print(
    "Final-test rows have NOT been loaded. "
    "They stay locked until notebook 17."
)
split_summary


Final-test rows have NOT been loaded. They stay locked until notebook 17.


,split,rows,positive_count,positive_rate
0,model_training,41991,3771,0.089805
1,threshold_validation,13998,1257,0.089799


In [7]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

decision_tree_preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numeric",
            numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

print("Decision Tree preprocessing created.")

Decision Tree preprocessing created.


In [8]:
def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    """
    Evaluate binary predictions created from predicted
    probabilities.

    Parameters
    ----------
    y_true:
        True binary labels.

    y_proba:
        Predicted probabilities for class 1.

    threshold:
        Probability threshold used to convert probabilities
        into class predictions.

    model_name:
        Name included in the result table.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp

    actual_positive = tp + fn
    actual_negative = tn + fp

    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    results = {
        "model": model_name,
        "threshold": float(threshold),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,

        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),

        "auroc": roc_auc_score(
            y_true,
            y_proba
        ),

        "auprc": average_precision_score(
            y_true,
            y_proba
        ),

        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),

        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),

        "predicted_positive": int(
            predicted_positive
        ),

        "predicted_negative": int(
            predicted_negative
        ),

        "predicted_positive_rate": (
            predicted_positive_rate
        ),

        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }

    return results

In [9]:
def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    """
    Create a labelled confusion matrix from predicted
    probabilities.
    """

    y_pred = (
        np.asarray(y_proba) >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )

In [10]:
def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    """
    Calculate model performance across a range of
    probability thresholds.
    """

    if thresholds is None:
        thresholds = np.round(
            np.arange(
                0.01,
                0.951,
                0.01
            ),
            2
        )

    results = [
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ]

    return pd.DataFrame(results)

In [11]:
def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    """
    Select the threshold with the lowest false-positive
    rate among thresholds achieving the required recall.

    Selection rule
    --------------
    1. Recall must be at least min_recall.
    2. Minimise false-positive rate.
    3. If tied, choose the highest threshold.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    false_positive_rates, recalls, thresholds = (
        roc_curve(
            y_true,
            y_proba,
            drop_intermediate=False
        )
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": (
            false_positive_rates
        ),
        "specificity": (
            1 - false_positive_rates
        )
    })

    candidate_table = candidate_table[
        np.isfinite(
            candidate_table["threshold"]
        )
    ].copy()

    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            "No threshold achieved recall >= "
            f"{min_recall:.2f}."
        )

    eligible_candidates = (
        eligible_candidates
        .sort_values(
            by=[
                "false_positive_rate",
                "threshold"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        eligible_candidates
        .iloc[0]["threshold"]
    )

    selected_metrics = (
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=selected_threshold,
            model_name=model_name
        )
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )

In [12]:
RECALL_TARGET = 0.80

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(
    f"Required validation recall: "
    f"{RECALL_TARGET:.0%}"
)

print(
    "Cross-validation folds:",
    cross_validation.n_splits
)

Required validation recall: 80%
Cross-validation folds: 5


In [13]:
decision_tree_param_distributions = {
    "model__criterion": [
        "gini",
        "entropy",
        "log_loss"
    ],

    "model__max_depth": [
        3,
        4,
        5,
        6,
        8,
        10,
        12,
        None
    ],

    "model__min_samples_split": [
        20,
        50,
        100,
        200,
        500
    ],

    "model__min_samples_leaf": [
        10,
        25,
        50,
        100,
        200,
        400
    ],

    "model__max_features": [
        None,
        "sqrt",
        "log2",
        0.5,
        0.75
    ],

    "model__max_leaf_nodes": [
        None,
        15,
        31,
        63,
        127
    ],

    "model__class_weight": [
        None,
        "balanced",
        {0: 1, 1: 2},
        {0: 1, 1: 4},
        {0: 1, 1: 6}
    ],

    "model__ccp_alpha": [
        0.0,
        0.00001,
        0.00005,
        0.0001,
        0.0005,
        0.001
    ]
}

In [14]:
decision_tree_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            decision_tree_preprocess
        ),
        (
            "model",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

decision_tree_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default

In [15]:
decision_tree_search = RandomizedSearchCV(
    estimator=decision_tree_pipeline,

    param_distributions=(
        decision_tree_param_distributions
    ),

    n_iter=150,

    scoring="average_precision",

    cv=cross_validation,

    refit=True,

    n_jobs=-1,

    verbose=2,

    random_state=42,

    return_train_score=True,

    error_score="raise"
)


decision_tree_search.fit(
    X_model_train,
    y_model_train
)

Fitting 5 folds for each of 150 candidates, totalling 750 fits
[CV] END model__ccp_alpha=1e-05, model__class_weight=balanced, model__criterion=log_loss, model__max_depth=5, model__max_features=0.5, model__max_leaf_nodes=None, model__min_samples_leaf=25, model__min_samples_split=200; total time=   0.3s
[CV] END model__ccp_alpha=1e-05, model__class_weight=balanced, model__criterion=log_loss, model__max_depth=5, model__max_features=0.5, model__max_leaf_nodes=None, model__min_samples_leaf=25, model__min_samples_split=200; total time=   0.4s
[CV] END model__ccp_alpha=1e-05, model__class_weight=balanced, model__criterion=log_loss, model__max_depth=5, model__max_features=0.5, model__max_leaf_nodes=None, model__min_samples_leaf=25, model__min_samples_split=200; total time=   0.4s
[CV] END model__ccp_alpha=1e-05, model__class_weight=balanced, model__criterion=log_loss, model__max_depth=5, model__max_features=0.5, model__max_leaf_nodes=None, model__min_samples_leaf=25, model__min_samples_split=2

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__ccp_alpha': [0.0, 1e-05, ...], 'model__class_weight': [None, 'balanced', ...], 'model__criterion': ['gini', 'entropy', ...], 'model__max_depth': [3, 4, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",150
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_

In [16]:
print("Best Decision Tree parameters:")

for parameter, value in (
    decision_tree_search
    .best_params_
    .items()
):
    print(f"{parameter}: {value}")


print("\nBest cross-validation AUPRC:")

print(
    decision_tree_search.best_score_
)

Best Decision Tree parameters:
model__min_samples_split: 20
model__min_samples_leaf: 400
model__max_leaf_nodes: 127
model__max_features: 0.75
model__max_depth: None
model__criterion: log_loss
model__class_weight: {0: 1, 1: 4}
model__ccp_alpha: 5e-05

Best cross-validation AUPRC:
0.13045126598973986


In [17]:
decision_tree_cv_results = pd.DataFrame(
    decision_tree_search.cv_results_
)


decision_tree_cv_results_selected = (
    decision_tree_cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",

            "param_model__criterion",
            "param_model__max_depth",
            "param_model__min_samples_split",
            "param_model__min_samples_leaf",
            "param_model__max_features",
            "param_model__max_leaf_nodes",
            "param_model__class_weight",
            "param_model__ccp_alpha"
        ]
    ]
    .sort_values(
        "rank_test_score"
    )
    .reset_index(drop=True)
)


decision_tree_cv_results_selected[
    "train_validation_auprc_gap"
] = (
    decision_tree_cv_results_selected[
        "mean_train_score"
    ]
    -
    decision_tree_cv_results_selected[
        "mean_test_score"
    ]
)


decision_tree_cv_results_selected.to_csv(
    OUTPUT_DIR
    / "decision_tree_cv_results.csv",
    index=False
)


decision_tree_cv_results_selected.head(20)

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,param_model__criterion,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf,param_model__max_features,param_model__max_leaf_nodes,param_model__class_weight,param_model__ccp_alpha,train_validation_auprc_gap
0,1,0.130451,0.004246,0.145662,0.002367,log_loss,None,20,400,0.75,127,"{0: 1, 1: 4}",0.00005,0.015211
1,2,0.130286,0.005664,0.140510,0.002076,gini,10,200,400,None,31,"{0: 1, 1: 6}",0.00001,0.010224
2,3,0.129971,0.004723,0.137997,0.001514,gini,6,500,400,0.75,63,"{0: 1, 1: 2}",0.00005,0.008026
3,4,0.129743,0.005482,0.143001,0.001355,gini,None,50,200,0.75,31,"{0: 1, 1: 4}",0.00005,0.013257
4,5,0.129647,0.006141,0.144766,0.002375,gini,None,20,400,None,63,"{0: 1, 1: 4}",0.00010,0.015119
5,6,0.129540,0.005709,0.140216,0.002198,entropy,None,100,400,None,31,"{0: 1, 1: 2}",0.00001,0.010676
6,7,0.129492,0.004756,0.156085,0.002505,gini,12,20,200,0.5,127,"{0: 1, 1: 2}",0.00000,0.026593
7,8,0.129344,0.005687,0.147045,0.001754,gini,12,100,400,0.75,127,balanced,0.00000,0.017701
8,9,0.129308,0.007032,0.141050,0.001659,log_loss,12,100,400,0.5,127,"{0: 1, 1: 4}",0.00010,0.011742
9,10,0.129307,0.006723,0.139393,0.001023,log_loss,6,200,400,None,31,"{0: 1, 1: 4}",0.00001,0.010086


In [18]:
decision_tree_top_candidates = (
    decision_tree_cv_results_selected
    .head(20)
    .copy()
)

display_columns = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "train_validation_auprc_gap",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__min_samples_split",
    "param_model__max_leaf_nodes",
    "param_model__class_weight",
    "param_model__ccp_alpha"
]

decision_tree_top_candidates[
    display_columns
]

,rank_test_score,mean_test_score,std_test_score,mean_train_score,train_validation_auprc_gap,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__max_leaf_nodes,param_model__class_weight,param_model__ccp_alpha
0,1,0.130451,0.004246,0.145662,0.015211,None,400,20,127,"{0: 1, 1: 4}",0.00005
1,2,0.130286,0.005664,0.140510,0.010224,10,400,200,31,"{0: 1, 1: 6}",0.00001
2,3,0.129971,0.004723,0.137997,0.008026,6,400,500,63,"{0: 1, 1: 2}",0.00005
3,4,0.129743,0.005482,0.143001,0.013257,None,200,50,31,"{0: 1, 1: 4}",0.00005
4,5,0.129647,0.006141,0.144766,0.015119,None,400,20,63,"{0: 1, 1: 4}",0.00010
5,6,0.129540,0.005709,0.140216,0.010676,None,400,100,31,"{0: 1, 1: 2}",0.00001
6,7,0.129492,0.004756,0.156085,0.026593,12,200,20,127,"{0: 1, 1: 2}",0.00000
7,8,0.129344,0.005687,0.147045,0.017701,12,400,100,127,balanced,0.00000
8,9,0.129308,0.007032,0.141050,0.011742,12,400,100,127,"{0: 1, 1: 4}",0.00010
9,10,0.129307,0.006723,0.139393,0.010086,6,400,200,31,"{0: 1, 1: 4}",0.00001


In [19]:
best_decision_tree_model = (
    decision_tree_search
    .best_estimator_
)

best_decision_tree_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['gender','race_group','age_group',...,'number_emergency', 'number_inpatient','number_diagnoses']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifyi

In [20]:
fitted_tree = (
    best_decision_tree_model
    .named_steps["model"]
)

tree_structure_summary = pd.Series({
    "tree_depth": fitted_tree.get_depth(),
    "number_of_leaves": fitted_tree.get_n_leaves(),
    "number_of_tree_nodes": (
        fitted_tree.tree_.node_count
    )
})

tree_structure_summary

tree_depth               11
number_of_leaves         70
number_of_tree_nodes    139
dtype: int64

In [21]:
tree_structure_summary.to_csv(
    OUTPUT_DIR
    / "decision_tree_structure_summary.csv",
    header=["value"]
)

In [22]:
y_val_proba_decision_tree = (
    best_decision_tree_model
    .predict_proba(X_val)[:, 1]
)


print(
    "Minimum validation probability:",
    y_val_proba_decision_tree.min()
)

print(
    "Maximum validation probability:",
    y_val_proba_decision_tree.max()
)

print(
    "Number of unique validation probabilities:",
    np.unique(
        y_val_proba_decision_tree
    ).size
)

Minimum validation probability: 0.07322654462242563
Maximum validation probability: 0.5560882070949185
Number of unique validation probabilities: 69


In [23]:
decision_tree_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_decision_tree,
        threshold=0.5,
        model_name=(
            "Decision Tree validation default"
        )
    )
)

pd.Series(
    decision_tree_val_default_results
)

model                                          Decision Tree validation default
threshold                                                                   0.5
accuracy                                                               0.905272
precision                                                              0.319372
recall                                                                 0.048528
specificity                                                            0.989797
false_positive_rate                                                    0.010203
false_negative_rate                                                    0.951472
f1                                                                     0.084254
f2                                                                      0.05844
auroc                                                                  0.608094
auprc                                                                  0.138219
brier_score                             

In [24]:
(
    decision_tree_selected_threshold,
    decision_tree_val_selected_results,
    decision_tree_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_decision_tree,
    min_recall=RECALL_TARGET,
    model_name=(
        "Decision Tree validation selected"
    )
)


print(
    "Selected Decision Tree threshold:"
)

print(
    decision_tree_selected_threshold
)


pd.DataFrame([
    decision_tree_val_default_results,
    decision_tree_val_selected_results
])

Selected Decision Tree threshold:
0.2153846153846154


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Decision Tree validation default,0.500000,0.905272,0.319372,0.048528,0.989797,0.010203,0.951472,0.084254,0.058440,...,0.138219,0.119389,12611,130,1196,61,191,13807,0.013645,3.131148
1,Decision Tree validation selected,0.215385,0.351550,0.102400,0.801114,0.307197,0.692803,0.198886,0.181589,0.338783,...,0.138219,0.119389,3914,8827,250,1007,9834,4164,0.702529,9.765641


In [25]:
decision_tree_eligible_thresholds.head(20)

,threshold,recall,false_positive_rate,specificity
0,0.215385,0.801114,0.692803,0.307197
1,0.206612,0.811456,0.703634,0.296366
2,0.204807,0.827367,0.725139,0.274861
3,0.202532,0.836913,0.739110,0.260890
4,0.201285,0.846460,0.757476,0.242524
5,0.194175,0.861575,0.776077,0.223923
6,0.183381,0.869531,0.792559,0.207441
7,0.180113,0.878282,0.802763,0.197237
8,0.170704,0.894988,0.819559,0.180441
9,0.163855,0.900557,0.837925,0.162075


In [26]:
decision_tree_threshold_sweep = (
    threshold_sweep(
        y_true=y_val,
        y_proba=y_val_proba_decision_tree,
        model_name=(
            "Decision Tree validation"
        )
    )
)


decision_tree_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "decision_tree_validation_threshold_sweep.csv",
    index=False
)


decision_tree_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f1",
        "f2",
        "true_positive",
        "false_positive",
        "false_negative",
        "predicted_positive_rate"
    ]
]

,threshold,recall,precision,specificity,false_positive_rate,f1,f2,true_positive,false_positive,false_negative,predicted_positive_rate
0,0.01,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
1,0.02,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
2,0.03,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
3,0.04,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
4,0.05,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
90,0.91,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
91,0.92,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
92,0.93,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
93,0.94,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0


In [27]:
decision_tree_validation_comparison = (
    pd.DataFrame([
        decision_tree_val_default_results,
        decision_tree_val_selected_results
    ])
)


comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "brier_score",
    "accuracy",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "false_negative_rate",
    "f1",
    "f2",
    "true_positive",
    "true_negative",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]


decision_tree_validation_comparison = (
    decision_tree_validation_comparison[
        comparison_columns
    ]
)


decision_tree_validation_comparison.to_csv(
    OUTPUT_DIR
    / "decision_tree_validation_results.csv",
    index=False
)


decision_tree_validation_comparison

,model,threshold,auprc,auroc,brier_score,accuracy,recall,precision,specificity,false_positive_rate,false_negative_rate,f1,f2,true_positive,true_negative,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Decision Tree validation default,0.500000,0.138219,0.608094,0.119389,0.905272,0.048528,0.319372,0.989797,0.010203,0.951472,0.084254,0.058440,61,12611,130,1196,0.013645,3.131148
1,Decision Tree validation selected,0.215385,0.138219,0.608094,0.119389,0.351550,0.801114,0.102400,0.307197,0.692803,0.198886,0.181589,0.338783,1007,3914,8827,250,0.702529,9.765641


In [28]:
decision_tree_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_decision_tree,
        threshold=(
            decision_tree_selected_threshold
        )
    )
)


decision_tree_val_selected_cm.to_csv(
    OUTPUT_DIR
    / (
        "decision_tree_validation_"
        "confusion_matrix.csv"
    )
)


decision_tree_val_selected_cm

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,3914,8827
Actual readmitted,250,1007


In [29]:
y_val_pred_decision_tree = (
    y_val_proba_decision_tree
    >= decision_tree_selected_threshold
).astype(int)


print(
    classification_report(
        y_val,
        y_val_pred_decision_tree,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)

                precision    recall  f1-score   support

Not readmitted       0.94      0.31      0.46     12741
    Readmitted       0.10      0.80      0.18      1257

      accuracy                           0.35     13998
     macro avg       0.52      0.55      0.32     13998
  weighted avg       0.86      0.35      0.44     13998



## Freeze final development-stage artifacts

In [30]:

RECALL_TARGETS = [0.70, 0.75, 0.80, 0.85, 0.90]

threshold_rows = []

for recall_target in RECALL_TARGETS:
    selected_threshold, selected_metrics, _ = (
        choose_threshold_for_minimum_recall(
            y_true=y_val,
            y_proba=y_val_proba_decision_tree,
            min_recall=recall_target,
            model_name="Decision Tree validation"
        )
    )

    threshold_rows.append({
        "target_recall": recall_target,
        "selected_threshold": selected_threshold,
        "validation_recall": selected_metrics["recall"],
        "validation_precision": selected_metrics["precision"],
        "validation_specificity": selected_metrics["specificity"],
        "validation_false_positive_rate": selected_metrics["false_positive_rate"],
        "validation_f2": selected_metrics["f2"],
        "validation_true_positive": selected_metrics["true_positive"],
        "validation_false_negative": selected_metrics["false_negative"],
        "validation_false_positive": selected_metrics["false_positive"],
        "validation_true_negative": selected_metrics["true_negative"],
        "validation_flagged_rate": selected_metrics["predicted_positive_rate"],
    })

selected_thresholds = pd.DataFrame(threshold_rows)

selected_thresholds.to_csv(
    OUTPUT_DIR / "selected_validation_thresholds.csv",
    index=False
)

validation_predictions = pd.DataFrame({
    "row_position": val_idx,
    "y_true": np.asarray(y_val, dtype=int),
    "probability": np.asarray(y_val_proba_decision_tree, dtype=float),
})

validation_predictions.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

selected_thresholds

import joblib
joblib.dump(best_decision_tree_model, OUTPUT_DIR / "final_model.joblib")

with open(OUTPUT_DIR / "selected_hyperparameters.json", "w") as f:
    json.dump(decision_tree_search.best_params_, f, indent=2, default=str)

print("Saved frozen Decision Tree model and artifacts.")


Saved frozen Decision Tree model and artifacts.
